# lab 7: attention is all you need

scopul lucrării:

* să scriem mecanismul de atenție în numpy și să îl verificăm pe exemplul din curs
* să folosim masca de atenție
* să comparăm cum împart în tokeni trei modele un text în română
* să luăm probabilitățile pentru următorul token și să generăm text la diferite temperaturi


## pregătirea

instalăm bertviz și descărcăm modelele o dată.

In [1]:
%pip install -q bertviz transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

MODELE_TOKENIZER = ["gpt2", "bert-base-multilingual-cased",
                    "dumitrescustefan/bert-base-romanian-cased-v1"]
for nume in MODELE_TOKENIZER:
    AutoTokenizer.from_pretrained(nume)

AutoModelForCausalLM.from_pretrained("gpt2")
AutoModel.from_pretrained("dumitrescustefan/bert-base-romanian-cased-v1")
print("Gata: modelele sunt descărcate.")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

D:\pr\PycharmProjects\slm_llm\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vica\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (15 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Gata: modelele sunt descărcate.


## partea 1. atenția în numpy

### sarcina 1.1. softmax stabil numeric

scădem maximul de pe fiecare rând înainte de `exp`. așa numerele nu devin uriașe.

In [3]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

def softmax(s):
    e = np.exp(s - s.max(axis=-1, keepdims=True))   # subtract row max
    return e / e.sum(axis=-1, keepdims=True)        # divide by row sum

print(softmax(np.array([1.2, 2.8, 0.7])))

[0.152 0.755 0.092]


acum comparăm varianta naivă cu varianta noastră pentru scoruri mari.

In [4]:
s = np.array([1000.0, 1001.0])
print(np.exp(s) / np.exp(s).sum())   # naive
print(softmax(s))                     # ours

[nan nan]
[0.269 0.731]


C:\Temp\ipykernel_15024\170461400.py:2: RuntimeWarning: overflow encountered in exp
  print(np.exp(s) / np.exp(s).sum())   # naive
C:\Temp\ipykernel_15024\170461400.py:2: RuntimeWarning: invalid value encountered in divide
  print(np.exp(s) / np.exp(s).sum())   # naive


### sarcina 1.2. funcția de atenție

scorurile sunt Q·Kᵀ / √d_k. apoi softmax dă ponderile, iar ponderile înmulțite cu V dau rezultatul.

In [5]:
def attention(Q, K, V, mask=None):
    d_k = K.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)
    weights = softmax(scores)
    return weights @ V, weights

q = np.array([[2, 1, 0, 1]])                     # q for "aprobat"
K = np.array([[0.5, 1.0, 0.3, 0.4],              # k for "Banca"
              [2.0, 1.0, 0.5, 0.6],              # k for "creditul"
              [0.2, 0.6, 0.9, 0.4]])             # k for "clientului"
V = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [0.5, 0.5]])

z, w = attention(q, K, V)
print("ponderi:", w)
print("z:", z)

ponderi: [[0.152 0.755 0.092]]
z: [[0.199 0.801]]


verificare: ponderile sunt [0.152 0.755 0.092] și z = [0.199 0.801]. în curs a ieșit (0.195; 0.805) pentru că acolo ponderile au fost rotunjite.

### sarcina 1.3. toți tokenii deodată, cu mască

o propoziție de 4 tokeni, fiecare cu 8 numere. masca lasă fiecare token să vadă doar tokenii de dinainte și pe el însuși.

In [6]:
rng = np.random.default_rng(42)
X = rng.normal(size=(4, 8))                        # 4 tokens x 8 features
W_Q, W_K, W_V = (rng.normal(size=(8, 4)) for _ in range(3))

Q, K, V = X @ W_Q, X @ W_K, X @ W_V                # step 4 from the lesson
mask = np.tril(np.ones((4, 4), dtype=bool))        # True = token can be seen
print(mask)

Z, W = attention(Q, K, V, mask)
print(W)
print("Suma pe rânduri:", W.sum(axis=1))

[[ True False False False]
 [ True  True False False]
 [ True  True  True False]
 [ True  True  True  True]]
[[1.    0.    0.    0.   ]
 [0.648 0.352 0.    0.   ]
 [0.491 0.145 0.363 0.   ]
 [0.285 0.341 0.281 0.093]]
Suma pe rânduri: [1. 1. 1. 1.]


verificare: suma pe fiecare rând este 1, deasupra diagonalei sunt doar zerouri, primul rând este [1 0 0 0], al doilea este [0.648 0.352 0 0].

### sarcina 1.4. multi-head attention cu două capete

fiecare cap face atenția lui. lipim rezultatele și le înmulțim cu W_O, care le combină.

In [7]:
def multi_head(X, heads, W_O):
    # heads = list of (W_Q, W_K, W_V), one per head
    outs = [attention(X @ Wq, X @ Wk, X @ Wv)[0] for Wq, Wk, Wv in heads]
    concat = np.concatenate(outs, axis=-1)
    return concat @ W_O                 # matrix that mixes the heads

heads = [tuple(rng.normal(size=(8, 4)) for _ in range(3)) for _ in range(2)]
W_O = rng.normal(size=(8, 8))
print(multi_head(X, heads, W_O).shape)

(4, 8)


verificare: forma este (4, 8). fiecare cap dă 4 numere, după lipire avem 8, iar W_O păstrează 8.

## partea 2. tokenizarea

comparăm trei modele: gpt-2 (mai ales engleză), bert multilingv (104 limbi) și un bert făcut pentru română.

la gpt-2 literele ă, î, ș, ț apar ca simboluri ciudate, de exemplu Ä. simbolul Ġ înseamnă un spațiu înainte de token.

In [8]:
nume_modele = {
    "GPT-2":           "gpt2",
    "BERT multilingv": "bert-base-multilingual-cased",
    "BERT românesc":   "dumitrescustefan/bert-base-romanian-cased-v1",
}
tokenizere = {nume: AutoTokenizer.from_pretrained(m) for nume, m in nume_modele.items()}
fraze = ["Banca a aprobat creditul clientului.",
         "Împrumuturile restructurate au dobânzi variabile."]

for nume, tok in tokenizere.items():
    for f in fraze:
        t = tok.tokenize(f)
        print(f"{nume:16} {len(t):3}  {t}")

GPT-2             12  ['B', 'anca', 'Ġa', 'Ġa', 'pro', 'bat', 'Ġcredit', 'ul', 'Ġclient', 'ul', 'ui', '.']
GPT-2             20  ['Ã', 'İ', 'm', 'pr', 'um', 'ut', 'ur', 'ile', 'Ġrestruct', 'urate', 'Ġau', 'Ġdo', 'b', 'Ã¢', 'n', 'zi', 'Ġvari', 'ab', 'ile', '.']
BERT multilingv   10  ['Banca', 'a', 'ap', '##ro', '##bat', 'credit', '##ul', 'client', '##ului', '.']
BERT multilingv   15  ['Î', '##mp', '##rumu', '##turi', '##le', 'rest', '##ru', '##ctura', '##te', 'au', 'dob', '##ân', '##zi', 'variabile', '.']
BERT românesc      6  ['Banca', 'a', 'aprobat', 'creditul', 'clientului', '.']
BERT românesc      8  ['Împrum', '##uturile', 'restruct', '##urate', 'au', 'dobânzi', 'variabile', '.']


### sarcina 2.1. numărul de tokeni

In [9]:
for nume, tok in tokenizere.items():
    n1, n2 = (len(tok.tokenize(f)) for f in fraze)
    print(f"{nume}: fraza 1 are {n1} tokeni, fraza 2 are {n2} tokeni")

GPT-2: fraza 1 are 12 tokeni, fraza 2 are 20 tokeni
BERT multilingv: fraza 1 are 10 tokeni, fraza 2 are 15 tokeni
BERT românesc: fraza 1 are 6 tokeni, fraza 2 are 8 tokeni


rezultatele mele:

* GPT-2: fraza 1 are 12 tokeni, fraza 2 are 20 de tokeni
* BERT multilingv: fraza 1 are 10 tokeni, fraza 2 are 15 tokeni
* BERT românesc: fraza 1 are 6 tokeni, fraza 2 are 8 tokeni

bert-ul românesc lasă aproape toate cuvintele întregi. gpt-2 rupe cuvintele în bucăți mici, iar literele î și â le rupe chiar în octeți.

### sarcina 2.2. cât costă limba română?

In [10]:
gpt2 = tokenizere["GPT-2"]
n_ro = len(gpt2.tokenize("Banca a aprobat creditul clientului."))
n_en = len(gpt2.tokenize("The bank approved the client's loan."))
print(n_ro, n_en, round(n_ro / n_en, 2))

12 8 1.5


In [11]:
# cost for one million sentences at 1 dollar per million tokens
print("română:", n_ro * 1_000_000 / 1_000_000, "dolari")
print("engleză:", n_en * 1_000_000 / 1_000_000, "dolari")

română: 12.0 dolari
engleză: 8.0 dolari


### sarcina 2.3. diacriticele cu virgulă și cu sedilă

ș, ț cu virgulă sunt forma corectă. ş, ţ cu sedilă sunt forma veche. pentru calculator sunt caractere diferite.

In [12]:
ro = tokenizere["BERT românesc"]
print(ro.tokenize("știință și țară"))     # comma
print(ro.tokenize("ştiinţă şi ţară"))     # cedilla

['știință', 'și', 'țară']
['[UNK]', '[UNK]', '[UNK]']


## partea 3. generarea textului

folosim gpt-2, un model care are doar decoder. lucrăm în engleză, pentru că gpt-2 a învățat pe texte în engleză.

### sarcina 3.1. probabilitățile pentru următorul token

In [13]:
import torch
torch.manual_seed(42)

tok = tokenizere["GPT-2"]
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

def top10(model, tok, text):
    ids = tok(text, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(ids).logits[0, -1]       # scores for the LAST position
    print("Dimensiunea vectorului logits:", logits.shape)
    probs = torch.softmax(logits, dim=-1)
    print("Suma probabilităților:", probs.sum().item())
    top = torch.topk(probs, 10)
    for p, i in zip(top.values, top.indices):
        print(f"{tok.decode(int(i))!r:16} {p.item():.3f}")

top10(model, tok, "The bank approved the")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Dimensiunea vectorului logits: torch.Size([50257])
Suma probabilităților: 1.0000118017196655
' loan'          0.066
' deal'          0.066
' sale'          0.037
' transaction'   0.033
' plan'          0.025
' purchase'      0.024
' new'           0.019
' $'             0.019
' move'          0.016
' settlement'    0.012


### sarcina 3.2. bucla de generare, scrisă de mână

prezicem, alegem, adăugăm, repetăm.

* **greedy** ia mereu tokenul cu scorul cel mai mare
* altfel împărțim scorurile la temperatură și alegem aleator după probabilități

In [14]:
def genereaza(model, tok, text, n_tokeni=25, temperatura=1.0, greedy=False):
    ids = tok(text, return_tensors="pt").input_ids
    for _ in range(n_tokeni):
        with torch.no_grad():
            logits = model(ids).logits[0, -1]
        if greedy:
            urm = torch.argmax(logits)                             # best token
        else:
            probs = torch.softmax(logits / temperatura, dim=-1)    # temperature
            urm = torch.multinomial(probs, 1)[0]                   # random pick
        ids = torch.cat([ids, urm.view(1, 1)], dim=1)              # append token
    return tok.decode(ids[0])

text = "The bank approved the"
print("GREEDY:", genereaza(model, tok, text, greedy=True))
for T in [0.2, 0.7, 1.5]:
    for incercare in range(2):
        print(f"T = {T}:", genereaza(model, tok, text, temperatura=T))

GREEDY: The bank approved the loan in May, but the bank has not yet approved the loan.

The bank said it will not be able to


T = 0.2: The bank approved the loan for $1.5 billion.

The bank said it will continue to lend to the U.S. government


T = 0.2: The bank approved the deal, which will allow the company to pay $1.2 billion to settle claims of fraud and other violations.




T = 0.7: The bank approved the loan, with interest rates at 55 per cent.

"We were worried about the idea that the price of such a


T = 0.7: The bank approved the petition, and the bank has now submitted a petition to the Supreme Court.

While the bank and the district court declined


T = 1.5: The bank approved the NBC breast shake notes in 2011 prohibiting free worryob shows and digital generation program StarManTales prefollowocopy Night Brother


T = 1.5: The bank approved the antibaccNet Infinity party role in November to convince lazy banker former magistrate dissident Communist tried by Atltka Chambers woolshirt


ce am observat:

* greedy: textul este coerent, dar se repetă ("the bank has not yet approved the loan")
* T = 0.2: coerent, seamănă cu greedy, sună ca o știre
* T = 0.7: variat și încă destul de coerent
* T = 1.5: incoerent, apar cuvinte fără legătură și chiar cuvinte inventate

### sarcina 3.3. (opțional) un model pentru limba română

RoGPT2 este un gpt-2 mic antrenat pe texte în română. textul nu va fi perfect, ne interesează mecanismul.

In [15]:
tok_ro_gpt = AutoTokenizer.from_pretrained("readerbench/RoGPT2-base")
model_ro_gpt = AutoModelForCausalLM.from_pretrained("readerbench/RoGPT2-base")
model_ro_gpt.eval()
top10(model_ro_gpt, tok_ro_gpt, "Banca a aprobat")
print(genereaza(model_ro_gpt, tok_ro_gpt, "Banca a aprobat", temperatura=0.7))

config.json:   0%|          | 0.00/814 [00:00<?, ?B/s]

D:\pr\PycharmProjects\slm_llm\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vica\.cache\huggingface\hub\models--readerbench--RoGPT2-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/563 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/985k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/542k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

... (4 outputs left out)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: readerbench/RoGPT2-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dimensiunea vectorului logits: torch.Size([50257])
Suma probabilităților: 1.0000123977661133
' în'            0.108
' un'            0.075
','              0.060
' o'             0.050
'-'              0.049
' finanțarea'    0.013
' planul'        0.013
' pentru'        0.013
' la'            0.012
' și'            0.010


Banca a aprobat pentru prima dată o redenumire prevăzută de legea din 1985. Un grup de lucru a fost format în 1994 pentru a elabora un


## partea 4 (opțională). embeddings contextuale

același cuvânt "bancă" primește vectori diferiți în funcție de context. stratul 0 este embedding plus poziție, stratul 12 este ieșirea ultimului bloc.

### sarcina 4.1. vectorul lui "bancă" în patru propoziții

In [16]:
import torch.nn.functional as Fn

nume_ro = "dumitrescustefan/bert-base-romanian-cased-v1"
tok_bert = AutoTokenizer.from_pretrained(nume_ro)
bert = AutoModel.from_pretrained(nume_ro, output_hidden_states=True,
                                 attn_implementation="eager")
bert.eval()

def vector_cuvant(fraza, cuvant, strat):
    enc = tok_bert(fraza, return_tensors="pt")
    with torch.no_grad():
        out = bert(**enc)
    tokens = tok_bert.convert_ids_to_tokens(enc.input_ids[0])
    parti = tok_bert.tokenize(cuvant)       # word can have several tokens
    for i in range(len(tokens) - len(parti) + 1):
        if tokens[i:i + len(parti)] == parti:
            return out.hidden_states[strat][0, i:i + len(parti)].mean(dim=0)
    raise ValueError(f"Nu am găsit „{cuvant}” în: {fraza}")

fraze_banca = ["Am deschis un cont la bancă.",
               "Am luat un credit de la bancă.",
               "M-am așezat pe o bancă în parc.",
               "Bătrânul stă pe bancă lângă fântână."]

for strat in [0, 12]:
    vecs = torch.stack([vector_cuvant(f, "bancă", strat) for f in fraze_banca])
    sim = Fn.cosine_similarity(vecs.unsqueeze(1), vecs.unsqueeze(0), dim=-1)
    print(f"Stratul {strat}:")
    print(sim.numpy().round(3))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Stratul 0:
[[1.    0.971 0.971 0.961]
 [0.971 1.    1.    0.954]
 [0.971 1.    1.    0.954]
 [0.961 0.954 0.954 1.   ]]


Stratul 12:
[[1.    0.953 0.659 0.594]
 [0.953 1.    0.625 0.56 ]
 [0.659 0.625 1.    0.832]
 [0.594 0.56  0.832 1.   ]]


### sarcina 4.2. vizualizarea atenției cu bertviz

alegem stratul din lista de sus și dăm clic pe un token, ca să vedem la cine se uită.

In [17]:
from bertviz import head_view

bert_att = AutoModel.from_pretrained(nume_ro, output_attentions=True,
                                     attn_implementation="eager")
enc = tok_bert("Banca a aprobat creditul clientului.", return_tensors="pt")
with torch.no_grad():
    att = bert_att(**enc).attentions
tokens_att = tok_bert.convert_ids_to_tokens(enc.input_ids[0])
head_view(att, tokens_att)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<IPython.core.display.Javascript object>

ca să nu căutăm doar cu ochiul, găsim și cu cod capul în care "aprobat" se uită cel mai mult la "creditul".

In [18]:
i_apr = tokens_att.index("aprobat")
i_cred = [i for i, t in enumerate(tokens_att) if t.startswith("credit")][0]

best = max(((l, h) for l in range(len(att)) for h in range(att[0].shape[1])),
           key=lambda lh: att[lh[0]][0, lh[1], i_apr, i_cred].item())
l, h = best
print(tokens_att)
print(f"stratul {l + 1}, capul {h + 1}: aprobat -> creditul = {att[l][0, h, i_apr, i_cred]:.3f}")

['[CLS]', 'Banca', 'a', 'aprobat', 'creditul', 'clientului', '.', '[SEP]']
stratul 2, capul 4: aprobat -> creditul = 0.995


ce am găsit:

* în stratul 2, capul 4, "aprobat" se uită la "creditul" cu ponderea 0.995. aici capul pare să lege verbul de complementul lui.
* în stratul 7, capul 6, toți tokenii se uită în medie 92% la [SEP]. un astfel de cap nu îl putem interpreta, pare un fel de "loc de parcare" pentru atenție când nu e nimic util de văzut.

## întrebări

**1. (sarcina 1.1) ce ați obținut cu softmax naiv pentru [1000, 1001] și ce cu varianta voastră? de ce apare diferența? de ce scăderea maximului nu schimbă rezultatul corect?**

varianta naivă dă [nan nan], iar varianta noastră dă [0.269 0.731].

exp(1000) este prea mare pentru calculator și devine infinit. infinit împărțit la infinit dă nan. când scădem maximul, scorurile devin [-1, 0], iar exp de ele este mic și se calculează fără probleme.

rezultatul corect nu se schimbă, pentru că exp(s − m) = exp(s) / exp(m). același număr exp(m) apare și sus, și jos în fracție, deci se simplifică.

**2. (sarcina 1.3) de ce primul rând al matricei de ponderi mascate este [1, 0, 0, 0]? ce tokeni vede al treilea token?**

primul token nu are nimic înainte, masca îl lasă să se vadă doar pe el. toate celelalte scoruri devin minus infinit, iar exp(minus infinit) = 0. deci toată ponderea, adică 1, merge la el însuși.

al treilea token vede tokenii 1, 2 și 3. ponderile lui sunt [0.491 0.145 0.363 0], deci la al patrulea token nu se uită deloc.

**3. (sarcina 2.2) de câte ori mai mulți tokeni are fraza în română decât în engleză, la gpt-2? cât ar costa un milion de fraze în fiecare limbă?**

la gpt-2 fraza în română are 12 tokeni, iar cea în engleză are 8. raportul este 1.5, deci româna folosește cu 50% mai mulți tokeni.

un milion de fraze în română înseamnă 12 milioane de tokeni, deci 12 dolari. în engleză sunt 8 milioane de tokeni, deci 8 dolari. pe lângă cost, în aceeași fereastră de context încape mai puțin text în română.

**4. (sarcina 2.3) ce diferență ați observat între "știință" și "ştiinţă"? ce ar trebui făcut cu un set de texte românești înainte de a fi dat unui model?**

cu virgulă (ș, ț) bert-ul românesc dă ['știință', 'și', 'țară'], adică trei cuvinte întregi. cu sedilă (ş, ţ) dă ['[UNK]', '[UNK]', '[UNK]'], adică modelul nu recunoaște nimic. toate cele trei cuvinte se pierd.

înainte să dăm texte românești unui model, trebuie să le normalizăm: înlocuim ş cu ș și ţ cu ț (și la fel pentru literele mari Ş, Ţ).

**5. (sarcina 3.1) ce dimensiune are vectorul logits la gpt-2? ce reprezintă acest număr? care au fost primii trei tokeni și cu ce probabilități?**

vectorul logits are dimensiunea 50257. acesta este numărul de tokeni din vocabularul gpt-2. modelul dă câte un scor pentru fiecare token posibil.

primii trei tokeni: ' loan' cu 0.066, ' deal' cu 0.066 și ' sale' cu 0.037. probabilitățile sunt mici, pentru că multe continuări sunt posibile.

**6. (sarcina 3.2) de ce varianta greedy produce mereu același text? ce se întâmplă cu textul când temperatura crește și de ce?**

greedy ia mereu tokenul cu scorul maxim. nu are nimic aleator, deci pentru același început dă mereu același text.

temperatura împarte scorurile înainte de softmax. la T mic, diferențele dintre scoruri cresc și cel mai bun token ia aproape toată probabilitatea, deci textul seamănă cu greedy și se repetă. la T mare, diferențele scad și distribuția devine aproape uniformă, deci se aleg și tokeni puțin probabili. textul devine variat, dar apoi incoerent.

**7. (partea 4, opțional) ce similarități ați obținut între "bancă" din propozițiile 1 și 2, respectiv 1 și 3, la stratul 0 și la stratul 12? ce arată aceste numere?**

* stratul 0: între propozițiile 1 și 2 similaritatea este 0.971, între 1 și 3 tot 0.971
* stratul 12: între propozițiile 1 și 2 similaritatea este 0.953, între 1 și 3 este 0.659

la stratul 0 vectorul lui "bancă" este aproape același peste tot, contează doar poziția. la stratul 12 modelul a citit contextul: cele două propoziții despre bani rămân foarte apropiate (0.953), iar banca din parc s-a îndepărtat (0.659 și 0.625). propozițiile 3 și 4, ambele despre locul de șezut, sunt și ele apropiate între ele (0.832). deci la stratul 12 vectorul depinde de sensul cuvântului.